In [7]:
import ee
import time
from collections import defaultdict, Counter

ee.Authenticate()
ee.Initialize(project="remotesensinggi")

# AEZ Assets Folder to be exported
AEZ_FOLDER = "projects/ee-endikebede20223/assets/As_AEZs"
# # Output folder
EXPORT_FOLDER_INFERENCE = "AEZ_inference_stacks_Asia"

SCALE = 30

CRS = None

CROPLAND_ASSET = None

GLOBAL_FEATURES = {
    "dist_river": "projects/ee-endikebede20223/assets/Dist_Asia_river",
    "dist_canal":  "projects/ee-endikebede20223/assets/Asia_dist_to_canal_300m",
}

KNOWN_FEATURES = [
    "ERA5_vapor_stats",
    "ET_sum_mm", "PET_sum_mm",
    "EVI_max", "EVI_mean", "EVI_min",
    "GI_max", "GI_mean", "GI_min",
    "NDVI_max", "NDVI_mean", "NDVI_min",
    "NDWI_max", "NDWI_mean", "NDWI_min",
    "SMAP_mean", "VCI",
    "elevation_m", "slope_deg",
]

# Int16 scale factors
# Stored value = real value * factor
# Real value   = stored value / factor

SCALE_FACTORS = {
    "NDVI_max": 10000,
    "NDVI_mean": 10000,
    "NDVI_min": 10000,
    "EVI_max": 10000,
    "EVI_mean": 10000,
    "EVI_min": 10000,
    "NDWI_max": 10000,
    "NDWI_mean": 10000,
    "NDWI_min": 10000,
    "VCI": 10000,
    "GI_max": 100,
    "GI_mean": 100,
    "GI_min": 100,

    "SMAP_surface_mean": 10000,
    "SMAP_rootzone_mean": 10000,

    "ERA5_vapor_stats_svp_mean": 10000,
    "ERA5_vapor_stats_vpd_mean": 10000,

    "ET_sum_mm": 10,
    "PET_sum_mm": 10,

    "elevation_m": 1,
    "slope_deg": 100,

    "dist_river": 1,
    "dist_canal": 1,
}

INT16_MIN = -32768
INT16_MAX = 32767

# Asset discovery

def parse_name(name):
    body = name[3:] if name.startswith("NM_") else name

    for feat in KNOWN_FEATURES:
        if body.endswith("_" + feat):
            return body[: -len("_" + feat)], feat

    raise ValueError(f"Could not parse: {name}")


children = ee.data.listAssets({"parent": AEZ_FOLDER})["assets"]
print("Type counts:", Counter(a["type"] for a in children))

children = [a for a in children if a["type"] == "IMAGE"]

groups = defaultdict(dict)

for a in children:
    name = a["id"].split("/")[-1]
    aez, feat = parse_name(name)
    groups[aez][feat] = a["id"]

aez_list = sorted(groups.keys())

print(f"Discovered {len(groups)} AEZs")
print(f"Discovered {sum(len(v) for v in groups.values())} feature assets total")

if CROPLAND_ASSET is not None:
    cropland = ee.Image(CROPLAND_ASSET)
    cropland_mask = cropland.select(0).gt(0)
else:
    cropland_mask = None


def scale_to_int16(img, band_name, factor):
    return (
        img.select(band_name)
        .multiply(factor)
        .round()
        .clamp(INT16_MIN, INT16_MAX)
        .toInt16()
        .rename(band_name)
    )


def prepare_band(img, band_name):
    if band_name not in SCALE_FACTORS:
        raise ValueError(f"No scale factor defined for band: {band_name}")

    return scale_to_int16(img, band_name, SCALE_FACTORS[band_name])


def print_scale_table():
    print("\nBand scale factors:")
    print("stored_value = real_value * scale_factor")
    print("real_value = stored_value / scale_factor\n")

    for band, factor in sorted(SCALE_FACTORS.items()):
        print(f"{band}: {factor}")


# Global feature stack

global_bands = []

for band_name, asset_id in GLOBAL_FEATURES.items():
    img = ee.Image(asset_id).select(0).rename(band_name)
    global_bands.append(prepare_band(img, band_name))

global_stack = ee.Image.cat(global_bands)


# Build stacked variables for each AEZ

def build_stack(aez, asset_ids):
    bands = []

    for feat in KNOWN_FEATURES:
        if feat not in asset_ids:
            print(f"  warning: {aez} missing {feat}")
            continue

        img = ee.Image(asset_ids[feat])

        if feat == "ERA5_vapor_stats":
            img = img.select(["svp_mean", "vpd_mean"]).rename([
                "ERA5_vapor_stats_svp_mean",
                "ERA5_vapor_stats_vpd_mean",
            ])

            for band_name in [
                "ERA5_vapor_stats_svp_mean",
                "ERA5_vapor_stats_vpd_mean",
            ]:
                bands.append(prepare_band(img, band_name))

        elif feat == "SMAP_mean":
            img = img.select(
                ["SMAP_surface", "SMAP_rootzone"]
            ).rename(
                ["SMAP_surface_mean", "SMAP_rootzone_mean"]
            )

            for band_name in [
                "SMAP_surface_mean",
                "SMAP_rootzone_mean",
            ]:
                bands.append(prepare_band(img, band_name))

        else:
            img = img.select(0).rename(feat)
            bands.append(prepare_band(img, feat))

    if not bands:
        raise ValueError(f"No bands found for {aez}")

    stacked = ee.Image.cat([ee.Image.cat(bands), global_stack])
    return stacked


# Export stacked variable GeoTIFFs

print_scale_table()

inference_tasks = []

for aez in aez_list:
    asset_ids = groups[aez]

    stacked = build_stack(aez, asset_ids)

    if cropland_mask is not None:
        stacked = stacked.updateMask(cropland_mask)

    region = ee.Image(next(iter(asset_ids.values()))).geometry().bounds()

    export_kwargs = dict(
        image=stacked,
        description=f"AEZ_{aez}_stacked_variables_int16",
        folder=EXPORT_FOLDER_INFERENCE,
        fileNamePrefix=f"{aez}_stacked_variables_int16",
        region=region,
        scale=SCALE,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True},
    )

    if CRS is not None:
        export_kwargs["crs"] = CRS

    task = ee.batch.Export.image.toDrive(**export_kwargs)

    task.start()
    inference_tasks.append((aez, task.id))

    print(f"Queued Int16 stacked variables export for {aez}: {task.id}")
    time.sleep(0.5)

print(f"\n{len(inference_tasks)} Int16 stacked variable exports queued")

Type counts: Counter({'IMAGE': 16})
Discovered 1 AEZs
Discovered 16 feature assets total

Band scale factors:
stored_value = real_value * scale_factor
real_value = stored_value / scale_factor

ERA5_vapor_stats_svp_mean: 10000
ERA5_vapor_stats_vpd_mean: 10000
ET_sum_mm: 10
EVI_max: 10000
EVI_mean: 10000
EVI_min: 10000
GI_max: 100
GI_mean: 100
GI_min: 100
NDVI_max: 10000
NDVI_mean: 10000
NDVI_min: 10000
NDWI_max: 10000
NDWI_mean: 10000
NDWI_min: 10000
PET_sum_mm: 10
SMAP_rootzone_mean: 10000
SMAP_surface_mean: 10000
VCI: 10000
dist_canal: 1
dist_river: 1
elevation_m: 1
slope_deg: 100
Queued Int16 stacked variables export for AS_SubTropWarmSH: KOUWIFN5DPI25KTRHP2YMP42

1 Int16 stacked variable exports queued
